In [ ]:
# %pip install anthropic

# curl https://api.anthropic.com/v1/messages \
#         --header "x-api-key: sk-ant-..." \
#         --header "anthropic-version: 2023-06-01" \
#         --header "content-type: application/json" \
#         --data \
#     '{
#         "model": "claude-sonnet-4-6",
#         "max_tokens": 1024,
#         "messages": [
#             {"role": "user", "content": "Hello, world"}
#         ]
#     }'

# {
#     "model": "claude-sonnet-4-6",
#     "id": "msg_01X4FZsFyNT9UaYHvJJhjkjb",
#     "type": "message",
#     "role": "assistant",
#     "content": [
#         {
#             "type": "text",
#             "text": "Hello! 👋 How are you doing? Is there something I can help you with today?"
#         }
#     ],
#     "stop_reason": "end_turn",
#     "stop_sequence": null,
#     "stop_details": null,
#     "usage": {"input_jannijannijannijannijannijannijannijannijannijannijannikkuhljannijannikkuhljannijannijannijannijannijannijannijannijannijannijannijannijannijannijannijannijanni

# Setup

In [1]:
import os
from anthropic import Anthropic
from dotenv import load_dotenv, find_dotenv

import base64
import json

load_dotenv(find_dotenv())

client = Anthropic(
    api_key=os.environ.get("ANTHROPIC_API_KEY"),  # This is the default and can be omitted
)

In [2]:
# claude-opus-4-7
# claude-sonnet-4-6
# claude-haiku-4-5

# Count tokens in a Message
[Claude API Docs](https://platform.claude.com/docs/en/api/python/messages/count_tokens)

In [3]:
Opus4_7_message_tokens_count = client.messages.count_tokens(
    messages=[{
        "content": "Hello, world",
        "role": "user",
    }],
    model="claude-opus-4-7",
)

Opus4_6_message_tokens_count = client.messages.count_tokens(
    messages=[{
        "content": "Hello, world",
        "role": "user",
    }],
    model="claude-opus-4-6",
)

Sonnet4_6_message_tokens_count = client.messages.count_tokens(
    messages=[{
        "content": "Hello, world",
        "role": "user",
    }],
    model="claude-sonnet-4-6",
)

Haiku4_5_message_tokens_count = client.messages.count_tokens(
    messages=[{
        "content": "Hello, world",
        "role": "user",
    }],
    model="claude-haiku-4-5",
)

print("Opus 4.7:", Opus4_7_message_tokens_count.input_tokens)
print("Opus 4.6:", Opus4_6_message_tokens_count.input_tokens)
print("Sonnet 4.6:", Sonnet4_6_message_tokens_count.input_tokens)
print("Haiku 4.5:", Haiku4_5_message_tokens_count.input_tokens)

Opus 4.7: 16
Opus 4.6: 10
Sonnet 4.6: 10
Haiku 4.5: 10


# PDF Upload via API

In [4]:
def encode_pdf_to_base64(file_path):
    """Liest ein PDF ein und gibt den Base64-String zurück."""
    with open(file_path, "rb") as pdf_file:
        return base64.b64encode(pdf_file.read()).decode('utf-8')

# Claude API Batch

In [5]:
message_batch = client.messages.batches.create(
    requests=[{
        "custom_id": "testung-2",
        "params": {
            "max_tokens": 1024,
            "messages": [{
                "content": "Hello, world",
                "role": "user",
            }],
            "model": "claude-haiku-4-5",
        },
    }],
)
print(message_batch.id)

msgbatch_01FBaveHWze8d91JFrGjjwuA


In [10]:
import time

MESSAGE_BATCH_ID = message_batch.id

message_batch = None
while True:
    message_batch = client.messages.batches.retrieve(MESSAGE_BATCH_ID)
    if message_batch.processing_status == "ended":
        break

    print(f"Batch {MESSAGE_BATCH_ID} is still processing...")
    time.sleep(60)
print(message_batch)

Batch msgbatch_01FBaveHWze8d91JFrGjjwuA is still processing...
Batch msgbatch_01FBaveHWze8d91JFrGjjwuA is still processing...
MessageBatch(id='msgbatch_01FBaveHWze8d91JFrGjjwuA', archived_at=None, cancel_initiated_at=None, created_at=datetime.datetime(2026, 5, 25, 12, 28, 17, 741069, tzinfo=datetime.timezone.utc), ended_at=datetime.datetime(2026, 5, 25, 12, 30, 33, 76040, tzinfo=TzInfo(0)), expires_at=datetime.datetime(2026, 5, 26, 12, 28, 17, 741069, tzinfo=datetime.timezone.utc), processing_status='ended', request_counts=MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=1), results_url='https://api.anthropic.com/v1/messages/batches/msgbatch_01FBaveHWze8d91JFrGjjwuA/results', type='message_batch')


In [14]:
for batch in client.messages.batches.results(
    message_batch.id,
):
  print(batch)

MessageBatchIndividualResponse(custom_id='testung-2', result=MessageBatchSucceededResult(message=Message(id='msg_01QYfRWkWjSNohsDypVb5GwM', container=None, content=[TextBlock(citations=None, text='Hello! 👋 How can I help you today?', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=10, output_tokens=16, server_tool_use=None, service_tier='batch')), type='succeeded'))


In [ ]:
# Stream results file in memory-efficient chunks, processing one at a time
for result in client.messages.batches.results(
    message_batch.id,
):
    match result.result.type:
        case "succeeded":
            print(f"Success! {result.custom_id}")
            print(result)
        case "errored":
            if result.result.error.error.type == "invalid_request_error":
                # Request body must be fixed before re-sending request
                print(f"Validation error {result.custom_id}")
            else:
                # Request can be retried directly
                print(f"Server error {result.custom_id}")
        case "expired":
            print(f"Request expired {result.custom_id}")

Success! testung-2
MessageBatchIndividualResponse(custom_id='testung-2', result=MessageBatchSucceededResult(message=Message(id='msg_01QYfRWkWjSNohsDypVb5GwM', container=None, content=[TextBlock(citations=None, text='Hello! 👋 How can I help you today?', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=10, output_tokens=16, server_tool_use=None, service_tier='batch')), type='succeeded'))


In [19]:
client.messages.batches.results(MESSAGE_BATCH_ID)